

## Learning Objectives

By the end of this session, you will:
1. Implement two complementary inference strategies:
   - Multiple generator calls (parallel, refinement)
   - Single long output generation (extended CoT)
2. Integrate feedback information (reward models, LLM feedback)
3. Compare and analyze different approaches
4. Understand compute-accuracy trade-offs


## Setup

### Install Dependencies

In [1]:
# Install required packages
!pip install openai transformers torch datasets matplotlib seaborn numpy pandas tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 8.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.0/68.0 kB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.9/107.9 kB 8.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.9/73.9 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 767.7/767.7 kB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 183.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.8/494.8 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 94.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import os
import json
import random
import re
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from collections import defaultdict, Counter
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from itertools import product
from typing import List, Dict, Any, Optional, Tuple

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification, pipeline

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# Configure plotting
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Check device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# For CPU-only environments, we'll use smaller models
assert device != "cpu", "We need GPUs for this practice!"

/opt/conda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


### Models

#### Loading

In [3]:
# Model options - pick one!
MODEL_OPTIONS = {
    "small": "Qwen/Qwen3-0.6B",   # 600M params
    "medium": "Qwen/Qwen3-1.7B",  # 1.7B params
    "large": "Qwen/Qwen3-4B",     # 4B params
}


MODEL_SIZE = "small"
SELECTED_MODEL = MODEL_OPTIONS[MODEL_SIZE]


print(f"Loading {SELECTED_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(SELECTED_MODEL)
model = AutoModelForCausalLM.from_pretrained(SELECTED_MODEL)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

generator = pipeline(
  "text-generation",
  model=model,
  tokenizer=tokenizer,
  model_kwargs={"torch_dtype": torch.bfloat16},
  device_map="auto",
)

print("Model loaded!")

Loading Qwen/Qwen3-0.6B...


Device set to use cuda:0


Model loaded!


#### Boilerplate

In [4]:
def generate_text(model, prompt: str | list[dict[str, str]], max_tokens=100, temperature=0.7):
    """Helper function to generate text"""
    if isinstance(prompt, str):
        prompt = [
            {"role": "system", "content": "Please reason step by step, and put your final answer within \\boxed{}."},
            {"role": "user", "content": prompt},
        ]

    # 1. Convert chat-style prompt to string
    prompt_str = tokenizer.apply_chat_template(prompt, tokenize=False, add_special_tokens=True)

    # 2. Tokenize that string
    inputs = tokenizer(prompt_str, return_tensors="pt", padding=True)
    input_ids = inputs["input_ids"].to(model.device)
    attention_mask = inputs["attention_mask"].to(model.device)

    # 3. Generate
    result = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=max_tokens,
        temperature=temperature,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
    )

    # 4. Slice out only the generated portion
    generated = result[0, input_ids.shape[-1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)


def extract_answer(text: str) -> int:
    """Extract numerical answer from text"""
    numbers = re.findall(r'\b\d+\b', text)
    return int(numbers[-1]) if numbers else None


def plot(data: Dict[str, List[Dict[str, Any]]], title: str):
    """
    Plot line graphs for multiple datasets side by side.

    Args:
        data: Dictionary with dataset names as keys and list of dicts as values.
              Each dict should have 'n', 'accuracy', and 'method' keys.
        title: Overall title for the plot
    """
    # Set up the plotting style
    sns.set_style("whitegrid")

    fig, axes = plt.subplots(1, len(data), figsize=(6 * len(data), 5), sharex=True)

    if len(data) == 1:
        axes = [axes]

    for ax, (name, dataset) in zip(axes, data.items()):
        df = pd.DataFrame(dataset)
        sns.lineplot(data=df, x='n', y='accuracy', hue='method', marker='o', ax=ax)
        ax.set_title(name)
        ax.legend(title='Method')

    plt.suptitle(title, fontweight='bold')
    plt.tight_layout()
    plt.show()

# global model
generate_text(model, "Hi, How are you?", max_tokens=100, temperature=1.0)

'\n<think>\nOkay, the user just asked, "Hi, How are you?" So they\'re probably testing if I can respond properly. First, I need to acknowledge their greeting. Since I\'m a language model, my first step is to say "Hi!" in English. Then, I should respond to the casual question. I should keep the response friendly and conversational. I can mention something about being helpful, like "I\'m here to assist you" or "I\'m happy to'

### Data

We'll use a curated set of multiplication and math problems for our experiments.

#### Boilerplate

In [5]:
def generate_multiplication_data(digits1: int, digits2: int, size: Optional[int] = None) -> List[Dict]:
    """
    Generate multiplication data with inputs of specified number of digits.

    Args:
        digits1: Number of digits for the first input
        digits2: Number of digits for the second input
        size: If specified, randomly sample this many combinations from all possible pairs
              If None, generate all possible combinations

    Returns:
        List of dictionaries with format: {"problem": "What is A x B?", "inputs": [A, B], "answer": C}
    """
    # Generate ranges for each operand based on digit count
    min1 = 10**(digits1 - 1) if digits1 > 1 else 1
    max1 = 10**digits1 - 1

    min2 = 10**(digits2 - 1) if digits2 > 1 else 1
    max2 = 10**digits2 - 1

    # Generate all possible operands
    operands1 = list(range(min1, max1 + 1))
    operands2 = list(range(min2, max2 + 1))

    if size is None:
        # Generate all combinations
        data = []
        for op1 in operands1:
            for op2 in operands2:
                data.append({
                    "problem": f"What is {op1} x {op2}?",
                    "inputs": [op1, op2],
                    "answer": op1 * op2
                })
        return data
    else:
        # Randomly sample combinations
        total_combinations = len(operands1) * len(operands2)

        if size > total_combinations:
            print(f"Warning: Requested size ({size}) exceeds total combinations ({total_combinations})")
            size = total_combinations

        # Generate random samples
        data = []
        sampled_pairs = set()

        while len(data) < size:
            op1 = random.choice(operands1)
            op2 = random.choice(operands2)

            # Avoid duplicates
            if (op1, op2) not in sampled_pairs:
                sampled_pairs.add((op1, op2))
                data.append({
                    "problem": f"What is {op1} x {op2}?",
                    "inputs": [op1, op2],
                    "answer": op1 * op2
                })

        return data


#### Generate multiplication data

In [6]:
DATA_SIZE = 100
mul_data_4x5 = generate_multiplication_data(digits1=4, digits2=5, size=DATA_SIZE // 4)
mul_data_5x5 = generate_multiplication_data(digits1=5, digits2=5, size=DATA_SIZE // 4)
mul_data_5x6 = generate_multiplication_data(digits1=5, digits2=6, size=DATA_SIZE // 4)
mul_data_7x5 = generate_multiplication_data(digits1=7, digits2=5, size=DATA_SIZE // 4)

mul_data = mul_data_4x5 + mul_data_5x5 + mul_data_5x6 + mul_data_7x5

print(f"Generated {len(mul_data)} multiplication problems.")

Generated 100 multiplication problems.


#### Load MATH-500

In [7]:
math_dataset = load_dataset("HuggingFaceH4/MATH-500", split="test").take(50)
print(f"Loaded {len(math_dataset)} MATH-500 problems.")

Generating test split: 100%|██████████| 500/500 [00:00<00:00, 107579.36 examples/s]

Loaded 50 MATH-500 problems.


#### Test Data Dict

In [8]:
TEST_DATA = {
    "multiplication": mul_data,
    "math_50": math_dataset,
}

### Test

In [9]:
# Sample an example response
test_item = TEST_DATA["multiplication"][-1]
# test_item = TEST_DATA["math_50"][-1]

print(f"Test Item:\n\n {test_item['problem']}")
# print(f"Test Item:\n\n {test_item}")

test_response = generate_text(model, test_item["problem"])
print(f"Test Response:\n\n{test_response}")

Test Item:

 What is 8099076 x 37760?
Test Response:



Okay, so I need to calculate 8,099,076 multiplied by 37,760. Hmm, that's a big number. Let me think about how to approach this. I remember that multiplying large numbers can be done by breaking them down into smaller parts, maybe using the distributive property. Let me try that.

First, maybe I can write both numbers as products of smaller numbers. Let's see, 8,09


## Before You Start

**Model**

MODEL is already loaded, use `generate_text` to sample from the model. Here is an example:
```python
generate_text(MODEL, "Hi, How are you?", max_tokens=100, temperature=1.0)
```
The output is a string.

The model is instructed to generate its step-by-step reasoning and put its answer within `\boxed{}`, e.g. `\boxed{345}`.

There is also a method named `extract_answer` to extract the numerical answer from model outputs.

**Data**
TEST_DATA is a dict whose key is the dataset name and its value contains the examples. The datasets are:

  - `multiplication`: 100 examples stored as a list of dict with the following format:
```json
{
    "problem": "What is {A} x {B}?"
    "inputs": [A, B],
    "answer": C,
}
```
  - `MATH-500`: 50 math problems (see [here](https://huggingface.co/datasets/HuggingFaceH4/aime_2024)) that are stored as a hf Dataset object with the following format:
```json
{
    "id": ID,
    "problem": "Define...",
    "solution": "...",
    "answer": "X",
}
```

Note that we only need "problem" and "answer" from both datasets.

In [10]:
N_SAMPLES1 = 4 # @param {type:"integer"}
MAX_TOKENS1 = 512 # @param {type:"integer"}

TEMPERATURE1 = 0.6 # @param {type:"number"}
assert 0.0 <= TEMPERATURE1 <= 1.0, "temperature must be between 0.0 and 1.0"

# Part 1: Majority Voting

## Your task:
  1. **Implement parallel generation:** Given a problem, sample multiple responses from the model

  2. **Run paralllel generation on the test data:** Use the method implemented in the previous step to collect samples for the entire test data.

  3. **Implement majority voting for each dataset and report accuracy:** Set `n_samples=4`, `temperature=0.6` and `max_tokens=512`.

  4. **Vary `n_samples` from 1 to 32 and report the trend**

  5. **Bonus: How does the results change if we increase `temperature` to 1.0?**

  6. **Bonus: How about increasing `max_tokens` to 2048?**

## Considerations

  - The model does NOT always generate an answer for various reasons including (i) not following the instructions or (ii) reaching its context limit. Your code should work for these cases as well.


## Your code:

In [13]:
import re
from collections import Counter

def extract_answer(response):
    # find the boxxed thingy: \\boxed{}..
    import re

    # Handle escaped \\boxed{...} or just boxed{...}
    match = re.search(r"\\boxed\{([^\}]+)\}", response)
    if match:
        return match.group(1).strip()

def majority_vote_responses(model, test_data, n_samples=4, temperature=0.6, max_tokens=512):
    correct = 0
    total = 0

    for item in test_data:
        counter = Counter()

        for _ in range(n_samples):
            response = generate_text(model, item["problem"], temperature=temperature, max_tokens=max_tokens)
            answer = extract_number(response)
            if answer:
                print(f"Response \n\n: {answer}")
                counter[answer] += 1

        if counter:
            voted_answer, _ = counter.most_common(1)[0]
            if str(voted_answer) == str(item["answer"]):
                correct += 1
        total += 1

    accuracy = correct / total if total > 0 else 0
    return accuracy

majority_vote_responses(model, TEST_DATA["multiplication"], N_SAMPLES1, TEMPERATURE1, MAX_TOKENS1)

Response 

: 2000
Response 

: 16944000
Response 

: 278
Response 

: 6
Response 

: 4200
Response 

: 98
Response 

: 42
Response 

: 0
Response 

: 63
Response 

: 4000
Response 

: 465
Response 

: 4
Response 

: 98700
Response 

: 91086
Response 

: 267
Response 

: 98700
Response 

: 3
Response 

: 200
Response 

: 1400
Response 

: 1395
Response 

: 79


KeyboardInterrupt: 

### Parallel generation

In [14]:
import re
from tqdm import tqdm

def extract_answer(response):
    # find the boxxed thingy: \\boxed{}..
    import re

    # Handle escaped \\boxed{...} or just boxed{...}
    match = re.search(r"\\boxed\{([^\}]+)\}", response)
    if match:
        return match.group(1).strip()

def sample_solutions(problem: str, n_samples: int, max_tokens: int = 512, temperature: float = 1.0) -> list[str]:
    answers = []

    for _ in range(n_samples):
        response = generate_text(model, problem, temperature=temperature, max_tokens=max_tokens)
        print(f'Response: {response} \n \n \n \n')
        answer = extract_number(response)
        if answer:
            answers.append(answer)

    return answers


def run_parallel_generation(test_data, n_samples: int, max_tokens: int = 512, temperature: float = 1.0):
    sampled_responses = defaultdict(list)
    for dataset_name, dataset in test_data.items():
      for example in tqdm(dataset, desc=f"{dataset_name}"):
        solutions = sample_solutions(example["problem"], n_samples, max_tokens, temperature)
        sampled_responses[dataset_name].append({
            "example": example,
            "sampled_responses": sample_solutions(example["problem"], n_samples, max_tokens, temperature),
        })
    return sampled_responses


SAMPLED_RESPONSES1 = run_parallel_generation(TEST_DATA, N_SAMPLES1, MAX_TOKENS1, TEMPERATURE1)

multiplication:   0%|          | 0/100 [00:00<?, ?it/s]

Response: 

Okay, so I need to figure out what 2824 multiplied by 13278 is. Hmm, multiplying two large numbers can be tricky. Let me think about how to approach this. 

First, maybe I can break down the numbers into parts that I can handle individually. Let me see... Both numbers have a lot of digits, so breaking them down might help. Let me check if they have any common factors or if there's a way to simplify it. Let me check if 2824 and 13278 have any common factors. 

Starting with 2824. Let me try dividing by small primes. 2824 is even, so divide by 2: 2824 / 2 = 1412. Still even, divide by 2 again: 1412 / 2 = 706. Again by 2: 706 / 2 = 353. Now, 353... Hmm, 353 is a prime number. Let me check if 353 is divisible by 3: 3 + 5 + 3 = 11, which isn't divisible by 3. Next prime is 5, ends with 3, so no. 7? 7*50 is 350, so 353 - 350 = 3, so no. 11? Let's see: 3 - 5 + 3 = 1, not divisible by 11. 13? 13*27 is 351, so 353 - 351 = 2, not divisible. So 353 is prime. Therefore, 2824 factors in

multiplication:   1%|          | 1/100 [01:02<1:42:35, 62.18s/it]

Response: 

Okay, so I need to calculate 2824 multiplied by 13278. Hmm, multiplying two large numbers can be tricky. Let me think about how to approach this. Maybe breaking it down into smaller parts would help? Let me try to recall if there's a method or formula for multiplying large numbers. Oh, right! I remember that breaking down the numbers into smaller components might make the multiplication easier. For example, sometimes you can split the multiplication into parts that are easier to compute individually.

Let me write down the numbers again to visualize better: 2824 and 13278. Let me check if these numbers have any common factors or if they can be simplified. Well, 2824 and 13278... Let me check if they share any common divisors. First, let me see if they are both even numbers. 2824 ends with a 4, so yes, it's even. Similarly, 13278 ends with 8, so it's also even. Therefore, both numbers are divisible by 2. Let me divide both numbers by 2 to simplify the calculation.

2824 ÷ 2 

multiplication:   1%|          | 1/100 [01:16<2:07:00, 76.97s/it]


KeyboardInterrupt: 

### Majority Voting

In [ ]:
from collections import Counter

def majority_voting(sampled_responses: list[str]) -> str:
    if not sampled_responses:
        return None  # fallback if no valid responses
    counter = Counter(sampled_responses)
    voted_answer, _ = counter.most_common(1)[0]
    return voted_answer

def run_majority_voting(sampled_responses, n_samples: int = 0):
  accuracy = defaultdict(int)

  for dataset_name, examples in sampled_responses.items():
    is_corrects = []
    for example in examples:
      sampled_responses = example["sampled_responses"]
      if n_samples > 0:
        sampled_responses = sampled_responses[:n_samples]
      majority_answer = majority_voting(sampled_responses)

      is_correct = int(majority_answer == example["example"]["answer"])
      is_corrects.append(is_correct)

    accuracy[dataset_name] = np.mean(is_correct)

  return accuracy


maj_at_n1 = run_majority_voting(SAMPLED_RESPONSES1)
print(f"majority voting@{N_SAMPLES1} = {maj_at_n1}")

### Varying N in Majority Voting

In [ ]:
MAX_N_SAMPELS = 32 # @param {type:"integer"}
assert MAX_N_SAMPELS > 0 and (MAX_N_SAMPELS & (MAX_N_SAMPELS - 1)) == 0, "MAX_N_SAMPLES must be a power of 2 (for simplicty)"

In [ ]:
SAMPLED_RESPONSES = run_parallel_generation(TEST_DATA, MAX_N_SAMPLES, MAX_TOKENS1, TEMPERATURE1)

maj_at_n_data = defaultdict(list)

for n in range(np.log2(MAX_N_SAMPLES) + 1):
  n_samples = 2 ** n
  maj_at_n = run_majority_voting(SAMPLED_RESPONSES, n_samples)
  for dataset_name, accuracy in maj_at_n.items():
    maj_at_n[dataset_name].append(
        {"n": n_samples, "accuracy": accuracy, "method": "majority voting"}
    )

plot(maj_at_n, "majority voting")

# Part 2: Best-of-N


## Your task:
  1. **Implement Best-of-N strategy given a reward model:** Reuse the parallel generations from Part 1. Keep the same parameters as Part 1.

  2. **Vary `N` from 1 to 32 and make a comparison with Majority Voting**

## Your code:

In [16]:
REWARD_MODEL_NAME = "Skywork-Reward-V2-Llama-3.1-8B-40M" # @param ["Skywork-Reward-V2-Llama-3.1-8B-40M", "Skywork/Skywork-Reward-V2-Qwen3-8B", "Skywork-Reward-V2-Llama-3.2-3B", "Skywork-Reward-V2-Qwen3-4B"]

### Loading Reward Model

In [ ]:
# off-load model from GPU
del MODEL
torch.cuda.empty_cache()
gc.collect()

# load reward model
reward_tokenizer = AutoTokenizer.from_pretrained(REWARD_MODEL_NAME)
reward_model = AutoModelForSequenceClassification.from_pretrained(
    REWARD_MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map=device,
    attn_implementation="flash_attention_2",
    num_labels=1,
)

def get_score(problem: str, response: str, model, tokenizer):
  conversation = [
      {"role": "user", "content": problem},
      {"role": "assistant", "content": response},
  ]

  prompt = tokenizer.apply_chat_template(conversation, tokenize=False)
  # following the recommendation in provided example: https://huggingface.co/Skywork/Skywork-Reward-V2-Qwen3-4B#%F0%9F%93%9D-simple-example-in-transformers
  if tokenizer.bos_token is not None and prompt.startswith(tokenizer.bos_token):
    prompt = prompt[len(tokenizer.bos_token):]
    # think this actually make sure that we're getting from beginning of the sequence and not including everything from the start that might cause bad performance or whatever

  tokenized_prompt = tokenizer(prompt, return_tensors="pt").to(model.device)

  with torch.no_grad():
    score = model(tokenized_prompt).logits[0][0].item()

  return score

def get_batch_scores(samples: list[dict[str, Any]], example, model, tokenizer):
  batch_scores = []

  for sample in samples:
    # example is the problem in this case that we're passing it to get_score
    score = get_score(example, sample, model, tokenizer)
    batch_scores.append({"sample": sample, "score": score})

  return batch_scores

### Best-of-N

In [ ]:
def best_of_n(sampled_responses: list[str], example, model, tokenizer) -> str:
  # we should call get_batch_scores here
  batch_scores = get_batch_scores(sampled_responses, example, model, tokenizer)

  best_answer = None
  best_score = float("-inf")

  for score in batch_scores:
    if score["score"] > best_score:
      best_score = score["score"]
      best_answer = score["sample"]

  return best_answer

def run_best_of_n(sampled_responses, model, tokenizer, n_samples: int = 0):
  accuracy = defaultdict(int)

  for dataset_name, examples in sampled_responses.items():
    is_corrects = []
    for example in examples:
      sampled_responses = example["sampled_responses"]
      if n_samples > 0:
        sampled_responses = sampled_responses[:n_samples]
      best_answer = best_of_n(sampled_responses, example["example"], model, tokenizer)

      is_correct = int(best_answer == example["example"]["answer"])
      is_corrects.append(is_correct)

    accuracy[dataset_name] = np.mean(is_correct)

  return accuracy

best_of_n1 = run_best_of_n(SAMPLED_RESPONSES1, reward_model, reward_tokenizer, N_SAMPLES1)
print(f"best-of-{N_SAMPLES1} = {best_of_n1}")

### Varying N in Best-of-N

In [ ]:
best_of_n_data = defaultdict(list)

for n in range(np.log2(MAX_N_SAMPLES) + 1):
  n_samples = 2 ** n
  best_of_n = run_best_of_n(SAMPLED_RESPONSES, reward_model, reward_tokenizer, n_samples)
  for dataset_name, accuracy in maj_at_n.items():
    best_of_n[dataset_name].append(
        {"n": n_samples, "accuracy": accuracy, "method": "best-of-N"}
    )

plot(best_of_n_data, "Best-of-N")

# Part 3: Self-correction


## Your tasks:
  1. **Implement self-correction:** Given sampled responses from Part 1 (use the first response from parallel generations), prompt the same model for a self-verification and collect the new answer.

  2. **Measure the accuracy after self-correction and compare it with before**

  3. **Bonus: Plot accuracy as a function of the number of output tokens**


## Your code:

### Loading model (again)

In [ ]:
# off-load reward model from GPU
del reward_model
torch.cuda.empty_cache()
gc.collect()


print(f"Loading {SELECTED_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(SELECTED_MODEL)
model = AutoModelForCausalLM.from_pretrained(SELECTED_MODEL)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

generator = pipeline(
  "text-generation",
  model=model,
  tokenizer=tokenizer,
  model_kwargs={"torch_dtype": torch.bfloat16},
  device_map="auto",
)

print("Model loaded!")


### Self-correct

In [ ]:
# think max_tokens is too small and the boxed thingy is not captured correctly

def extract_answer(response):
    # find the boxxed thingy: \\boxed{}..
    import re

    # Handle escaped \\boxed{...} or just boxed{...}
    match = re.search(r"\\boxed\{([^\}]+)\}", response)
    if match:
        return match.group(1).strip()

    # Fallback: look for any number at end of line
    numbers = re.findall(r"\d+", response)
    return numbers[-1] if numbers else None

def generate_text_self_correct(model, problem, response, max_tokens=512, temperature=0.7):
    """Helper function to generate text to self correct"""

    # NOTE: prompt and answer were undefined, this fixes that
    prompt = [
        {
            "role": "system",
            "content": (
                f"Here's your answer for this question:\n{problem}\n\n"
                f"Your answer: {response}\n\n"
                f"Based on your answer, correct if anything was wrongly determined, reason step by step, "
                f"and put your final answer within \\boxed{{}}."
            )
        }
    ]

    # 1. Convert chat-style prompt to string
    prompt_str = tokenizer.apply_chat_template(prompt, tokenize=False, add_special_tokens=True)

    # 2. Tokenize that string
    inputs = tokenizer(prompt_str, return_tensors="pt", padding=True)
    input_ids = inputs["input_ids"].to(model.device)
    attention_mask = inputs["attention_mask"].to(model.device)

    # 3. Generate
    result = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=max_tokens,  # think max_tokens is too small → increased from 100 to 512
        temperature=temperature,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
    )

    # 4. Slice out only the generated portion
    generated = result[0, input_ids.shape[-1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)


def self_correct(problem: str, response: str, model):
  response = generate_text_self_correct(model, problem, response)
  return extract_answer(response)

def run_self_correct(sampled_responses, model):
  accuracy = defaultdict(int)

  for dataset_name, examples in sampled_responses.items():
    is_corrects = []
    for example in examples:
      response = example["sampled_responses"][0]
      modified_answer = self_correct(example["example"]["problem"], response, model)

      is_correct = int(modified_answer == example["example"]["answer"])
      is_corrects.append(is_correct)

    accuracy[dataset_name] = np.mean(is_correct)

  return accuracy


self_correct = run_self_correct(SAMPLED_RESPONSES1, model)
print(f"self-correct = {self_correct}")

## Reflection Questions:

1. Which strategy worked best for multiplication problems? Why?
2. How did compute budget affect your results?
3. What are the trade-offs between the two main approaches?
4. How would you extend these methods to harder problems?
5. What external information would be most helpful?